# Complex plane waves in one and two dimensions

## Prerequisites

See the [complex plane-wave lecture note](../../../docs/lecture-notes/waves/plane/complex-plane-waves/index.md). This laboratory assumes complex exponentials, angular wave number, angular frequency, and the discrete Fourier transform.

## Objectives

1. Evaluate unit-aware one- and two-dimensional complex plane waves.
2. Visualize real and imaginary components in one dimension.
3. Confirm the constant field magnitude.
4. Recover an on-grid two-dimensional wave vector by an independent FFT calculation.

The represented field is generic. Calling it a quantum wavefunction, electromagnetic field, displacement, or another physical observable requires additional model-specific contracts.


## Plane-wave convention

The maintained models use

$$
u(\mathbf r,t)=A\exp\left[i\left(\mathbf k\cdot\mathbf r-\omega t+\phi\right)\right],$$

where $A$ and $\phi$ are dimensionless, $\mathbf k$ has inverse-length units, and $\omega$ has inverse-time units. No dispersion relation between $\mathbf k$ and $\omega$ is imposed.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.units import (
    PhysicalUnit,
    ScalarQuantity,
    Unitless,
    VectorQuantity,
)
from projectkoios.physkit.waves.plane.one_dimensional.model import (
    PlaneWave1DModel,
    PlaneWave1DParameters,
)
from projectkoios.physkit.waves.plane.two_dimensional.model import (
    PlaneWave2DModel,
    PlaneWave2DParameters,
)

In [ ]:
domain_length = 10.0
sample_count = 256
x_coordinates = np.linspace(
    0.0,
    domain_length,
    sample_count,
    endpoint=False,
)
one_dimensional_model = PlaneWave1DModel(
    parameters=PlaneWave1DParameters(
        amplitude=ScalarQuantity(magnitude=1.0, unit=Unitless()),
        wave_number=ScalarQuantity(
            magnitude=2.0 * np.pi * 3.0 / domain_length,
            unit=PhysicalUnit(expression="1 / meter"),
        ),
        angular_frequency=ScalarQuantity(
            magnitude=2.0 * np.pi,
            unit=PhysicalUnit(expression="1 / second"),
        ),
        phase_offset=ScalarQuantity(magnitude=np.pi / 6.0, unit=Unitless()),
    )
)
one_dimensional_evaluation = one_dimensional_model.evaluate(
    positions=VectorQuantity(
        magnitude=x_coordinates,
        unit=PhysicalUnit(expression="meter"),
    ),
    time=ScalarQuantity(
        magnitude=0.25,
        unit=PhysicalUnit(expression="second"),
    ),
)
assert np.allclose(
    np.abs(one_dimensional_evaluation.field_values.magnitude),
    1.0,
    rtol=0.0,
    atol=1.0e-15,
)

In [ ]:
mode_x = 5
mode_y = 7
two_dimensional_model = PlaneWave2DModel(
    parameters=PlaneWave2DParameters(
        amplitude=ScalarQuantity(magnitude=1.0, unit=Unitless()),
        wave_vector=VectorQuantity(
            magnitude=np.array(
                [
                    2.0 * np.pi * mode_x / domain_length,
                    2.0 * np.pi * mode_y / domain_length,
                ]
            ),
            unit=PhysicalUnit(expression="1 / meter"),
        ),
        angular_frequency=ScalarQuantity(
            magnitude=0.0,
            unit=PhysicalUnit(expression="1 / second"),
        ),
        phase_offset=ScalarQuantity(magnitude=0.0, unit=Unitless()),
    )
)
two_dimensional_evaluation = two_dimensional_model.evaluate(
    x_coordinates=VectorQuantity(
        magnitude=x_coordinates,
        unit=PhysicalUnit(expression="meter"),
    ),
    y_coordinates=VectorQuantity(
        magnitude=x_coordinates,
        unit=PhysicalUnit(expression="meter"),
    ),
    time=ScalarQuantity(
        magnitude=0.0,
        unit=PhysicalUnit(expression="second"),
    ),
)

In [ ]:
spectrum = np.fft.fftshift(
    np.fft.fft2(two_dimensional_evaluation.field_values.magnitude) / sample_count**2
)
wave_numbers = np.fft.fftshift(
    2.0 * np.pi * np.fft.fftfreq(sample_count, d=domain_length / sample_count)
)
peak_y_index, peak_x_index = np.unravel_index(
    np.argmax(np.abs(spectrum)),
    spectrum.shape,
)
detected_wave_vector = np.array(
    [wave_numbers[peak_x_index], wave_numbers[peak_y_index]]
)
assert np.allclose(
    detected_wave_vector,
    two_dimensional_model.parameters.wave_vector.magnitude,
    rtol=0.0,
    atol=1.0e-12,
)
assert np.isclose(np.max(np.abs(spectrum)), 1.0)
detected_wave_vector

In [ ]:
figure, (line_axis, field_axis, spectrum_axis) = plt.subplots(
    nrows=1,
    ncols=3,
    figsize=(15.0, 4.5),
)
line_axis.plot(
    x_coordinates,
    one_dimensional_evaluation.field_values.magnitude.real,
    label="real part",
)
line_axis.plot(
    x_coordinates,
    one_dimensional_evaluation.field_values.magnitude.imag,
    linestyle="--",
    label="imaginary part",
)
line_axis.set(
    xlabel="position (m)",
    ylabel="unitless field",
    title="One-dimensional plane wave",
)
line_axis.grid(True)
line_axis.legend()

field_image = field_axis.imshow(
    two_dimensional_evaluation.field_values.magnitude.real,
    extent=(0.0, domain_length, 0.0, domain_length),
    origin="lower",
    cmap="RdBu_r",
    vmin=-1.0,
    vmax=1.0,
)
field_axis.set(
    xlabel="x (m)",
    ylabel="y (m)",
    title="Real part in physical space",
)
figure.colorbar(field_image, ax=field_axis, label="unitless field")

spectrum_image = spectrum_axis.imshow(
    np.log1p(np.abs(spectrum)),
    extent=(
        wave_numbers[0],
        wave_numbers[-1],
        wave_numbers[0],
        wave_numbers[-1],
    ),
    origin="lower",
    cmap="magma",
)
spectrum_axis.set(
    xlim=(-10.0, 10.0),
    ylim=(-10.0, 10.0),
    xlabel=r"$k_x$ (m$^{-1}$)",
    ylabel=r"$k_y$ (m$^{-1}$)",
    title="Log FFT magnitude",
)
figure.colorbar(spectrum_image, ax=spectrum_axis, label=r"$\log(1+|\widehat u|)$")
figure.tight_layout()
plt.show()

## Interpretation

The calculated 1D field follows the expected phase shift in space and time. Its magnitude remains the selected amplitude. The independent 2D FFT has one dominant peak at the requested periodic-grid wave vector.

The peak test is a discrete representation check. It does not identify the field as a physical quantum state or establish a dispersion relation.

## Exercises

1. Add a nonzero phase offset and predict both plots before execution.
2. Change the time and verify that the FFT peak position remains fixed while its phase changes.
3. Choose a wave vector between FFT grid points and examine spectral leakage.
4. Construct a superposition of two package-evaluated plane waves and identify both reciprocal-space peaks.
